#### <font color='#0000DD'><b> Welcome to the prototype demonstration for the paper *KRAFT -- A Knowledge-Graph-Based Resource Allocation Framework*
<font color='#0000DD'><b> This Jupyter Notebook walks you through the features of the prototype and framework. Marked in blue is contextual information and instructions.

# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from urllib.parse import quote, unquote
import pandas as pd
from datetime import timedelta

In [3]:
import sys  
import os
root = os.path.abspath('')
sys.path.insert(1, root + '/Business Process Optimization Competition 2023')
from simulator import Simulator
from problems import MinedProblem
sys.path.insert(1, root + '/src')
from KGPlanner import KGPlanner, Mode
from ProcessKnowledgeGraph import ProcessKnowledgeGraph
from SHACLAllocator import SHACLAllocator 
from SimulatorForDemonstration import SimulatorForDemonstration
from utils import *

# Mining Resource Expertise
<font color='#0000DD'><b>The following demonstrates how process mining techniques can be utilized to extract knowledge about resources, in this case expertise for specific loan goals derived from the RBI 1.2 "Case Types" from Pika et al., "Mining Resource Profiles from Event Logs" (2017).
First, the BPIC log is read from file and the respective metrics (cases completed per resource in total and further divided per loan goal) are calculated.
Then, expertise for specific loan goals is derived via a threshold from the RBI.

In [4]:
import pandas as pd
import numpy as np

In [5]:
log = pd.read_csv('./BPI Challenge 2017 - clean.csv')
min_date = log['Start Timestamp'].min()
max_date = log['Complete Timestamp'].max()
total_cases_per_resource = log.groupby(['Resource']).agg({'Case ID' : pd.Series.nunique})['Case ID']
cases_per_resource_per_loan_goal = log.groupby(['LoanGoal', 'Resource']).agg({'Case ID' : pd.Series.nunique}).unstack()['Case ID']
cases_per_resource_per_loan_goal_fraction = cases_per_resource_per_loan_goal.div(total_cases_per_resource.transpose()).fillna(0) # The RBI 1.2
experts = cases_per_resource_per_loan_goal_fraction > 0.35

<font color='#0000DD'><b> To translate this knowledge to graph, nodes for the resources and loan goals as well as appropriate `expertOn` edges are created:

In [6]:
helper_graph = ProcessKnowledgeGraph()
for loan_goal in experts.index:
    helper_graph.add_entity('LoanGoal', loan_goal)
    for resource in experts.columns:
        if experts.loc[loan_goal][resource]:
            helper_graph.add_entity('resource', resource)
            helper_graph.add_from_strings('resource:'+resource, 'expertOn', 'LoanGoal:'+loan_goal)

<font color='#0000DD'><b> We further determine seniority of the resources based on the number of case completions

In [7]:
seniorities = total_cases_per_resource.apply(lambda cpr : 'High' if cpr > 1500 else ('Medium' if cpr > 300 else 'Low'))
for resource, seniority in seniorities.items():
    helper_graph.add_entity('resource', resource) # Redundant for resources that are not experts on anything; will not produce duplicates
    helper_graph.add_entity('Seniority', seniority)
    helper_graph.add_from_strings('resource:'+resource, 'Seniority', 'Seniority:'+seniority)

<font color='#0000DD'><b>  You can have a look at the generated graph as follows. Note that you can put this viewer into fullscreen mode, and you can explore a node's neighbourhood by selecting a node and opening the sidebar.

In [8]:
draw_graph(helper_graph)

GraphWidget(layout=Layout(height='800px', width='100%'))

<font color='#0000DD'><b> Now write the knowledge to file for later use:

In [9]:
helper_graph.serialize(destination='./src/extension_knowledge/expertise_instances.ttl')

<Graph identifier=Nf956ac6dc96a446087429e9eabaca510 (<class 'ProcessKnowledgeGraph.ProcessKnowledgeGraph'>)>

# Example Simulation
<font color='#0000DD'><b>Execute the following three steps to run the example simulation. Don't rerun single steps.

### Planner & Allocator Setup
<font color='#0000DD'><b>The planner is the interface to the simulation frame, implementing a function to assign a set of open tasks.
The allocation actually manages the reasoning based on the encoded ontology and rules. In this example setup, we load additional knowledge as described in the paper's example use case.
Note that we reuse the knowledge earlier derived from Process Mining by loading the respective file.

In [27]:
planner = KGPlanner()
planner.allocator.load_extension(ontology_ext='./src/extension_knowledge/extension_ontology.ttl')
planner.allocator.load_extension(rules_ext='./src/extension_knowledge/sepOfConcerns_rules.ttl')
planner.allocator.load_extension(rules_ext='./src/extension_knowledge/riskAndSeniority_rules.ttl', instance_ext='./src/extension_knowledge/riskAndSeniority_instances.ttl')
planner.allocator.load_extension(rules_ext='./src/extension_knowledge/expertise_rules.ttl', instance_ext='./src/extension_knowledge/expertise_instances.ttl')

<font color='#0000DD'><b> At this point, and at any point when the simulation is paused, it is possible to manually add further knowledge. 
For example, let's manually define some expertises that have not been discovered by the process mining techniques: 

In [28]:
planner.graph.add_from_strings('resource:User_24', 'expertOn', 'LoanGoal:Remaining debt home')
planner.graph.add_from_strings('resource:User_99', 'expertOn', 'LoanGoal:Boat')
planner.graph.add_from_strings('resource:User_123', 'expertOn', 'LoanGoal:Extra spending limit')
planner.graph.add_from_strings('resource:User_56', 'expertOn', 'LoanGoal:Home improvement')

### Simulation Frame Setup
<font color='#0000DD'><b>The simulation frame and simulated process are taken from the BPOC 2023. We adapt the simulation numbers as to allow for more interesting allocation situations.

In [29]:
simulator = SimulatorForDemonstration(planner=planner, instance_file='./Business Process Optimization Competition 2023/BPI Challenge 2017 - instance 2.pickle')
problem = simulator.problem
# Increase Arrival Rate so also more resources are provided
problem.interarrival_time._scale *= 0.5
# Increase likelihood for assessing fraud, so separation of concerns actually triggers
problem.next_task_distribution['W_Validate application'] = list(map(lambda x : (x[0] * 0.8 + (0.2 if (x[1] == 'W_Assess potential fraud') else 0), x[1]), problem.next_task_distribution['W_Validate application']))

### Run the Simulation

<font color='#0000DD'><b>First, let's run just run the initial first task assignment of the simulation. The console logs show current allocation context (case, task, and activity as well as available and pool-allowed resources), the decision made by the system and the respective reasoning. 

In [13]:
simulator.run(0)


0 case-0 task-0: W_Complete application 
Resources Available: {'User_37', 'User_34', 'User_144', 'User_66', 'User_12'} 
Resources in Pool: {'User_144', 'User_66', 'User_34', 'User_37', 'User_12'}
Interrupt
Pausing Simulation at time 0. Will complete running current cycle. Interrupt again for immediate cancellation.
Interrupt
Pausing Simulation at time inf. Will complete running current cycle. Interrupt again for immediate cancellation.


<font color='#0000DD'><b>We can now investigate the current state of the allocation knowledge graph after one allocation decision. 
To ensure that we only see the resources currently present, we filter accordingly.  
This is a nice illustration of the relevant entities for one single decision (although some background knowledge entities are visible as well as they are already loaded).
Note that entities are color-coded by their type. 

In [14]:
draw_graph(planner.graph.subgraph_available_resources())

GraphWidget(layout=Layout(height='800px', width='100%'))

<font color='#0000DD'><b> Let's resume the simulation run until 1 hours of simulation time. This will take some time. However, the simulation can be paused at any moment using Jupyter's interrupt function (the interrupt button or pressing `i` twice while having the respective cell selected.) 

In [15]:
oldgraph = planner.graph
planner.graph = planner.graph.subgraph_available_resources()# TODO Cheeeky

In [16]:
simulator.run(1)

Resuming simulation at time 0. Remaining time to run: 1


(0.0,
 'COMPLETED: you completed 0 hours of simulated customer cases. 1 cases started. 0 cases run to completion.')

<font color='#0000DD'><b> While paused, we can add more knowledge, and also change the system mode to human-in-the-loop (HITL).
In HITL, the system prompts users for every allocation decision. We can then continue the simulation a little more in this mode.

In [17]:
planner.graph.add_entity('ApplicationType', 'Limit raise')
planner.graph.add_from_strings('resource:'+resource, 'expertOn', 'ApplicationType:Limit raise')
planner.mode = Mode.HUMAN_IN_THE_LOOP
simulator.run(1.1)


0.009824661898165984 case-1 task-1: W_Complete application 
Resources Available: {'User_37', 'User_34', 'User_144', 'User_66', 'User_12'} 
Resources in Pool: {'User_144', 'User_66', 'User_34', 'User_37', 'User_12'}
The following resources are available:
0: User_144, score: 0, considering: 
 
1: User_12, score: 0, considering: 
 
2: User_37, score: 0, considering: 
 
3: User_66, score: 0, considering: 
 
4: User_34, score: 0, considering: 
 
Type the index of the resource to select. Type -1 to not assign any resource.
Interrupt
Pausing Simulation at time 0.009824661898165984. Will complete running current cycle. Interrupt again for immediate cancellation.
Interrupt
Pausing Simulation at time inf. Will complete running current cycle. Interrupt again for immediate cancellation.


<font color='#0000DD'><b> Afterward, we can reset the mode and continue the simulation until hour 2.

In [18]:
planner.mode = Mode.AUTOMATED
simulator.run(2)

Resuming simulation at time 0.009824661898165984. Remaining time to run: 2


(0.0,
 'COMPLETED: you completed 0 hours of simulated customer cases. 2 cases started. 1 cases run to completion.')

<font color='#0000DD'><b> Finally, we can have a look at the whole final graph.

In [19]:
draw_graph(planner.graph)

GraphWidget(layout=Layout(height='800px', width='100%'))

# Dev

In [20]:
raise Exception('Don\'t just jump into dev')


KeyboardInterrupt



In [26]:
%load_ext line_profiler

In [ ]:
%lprun -f simulator.run simulator.run(0)


0.007613186589093906 case-2 task-2: W_Complete application 
Resources Available: {'User_107', 'User_104', 'User_56'} 
Resources in Pool: {'User_104', 'User_56'}


In [ ]:
float('inf')

In [ ]:
x = simulator.problem.data_types['ApplicationType']
sorted(zip(x._weights, x._values))

In [ ]:
x = simulator.problem.data_types['LoanGoal']
sorted(zip(x._weights, x._values))

In [ ]:
print(planner.graph.serialize(format='n3'))

In [ ]:
from ProcessKnowledgeGraph import ProcessKnowledgeGraph

In [ ]:
test_graph_data = '''
@prefix ApplicationType: <http://example.org/instances/ApplicationTypes/> .
@prefix LoanGoal: <http://example.org/instances/LoanGoals/> .
@prefix activity: <http://example.org/instances/activitys/> .
@prefix case: <http://example.org/instances/cases/> .
@prefix task: <http://example.org/instances/tasks/> .
@prefix relation: <http://example.org/relations/> .
@prefix resource: <http://example.org/instances/resources/> .
@prefix four_eyes: <http://example.org/instances/four_eyes/> .
@prefix type: <http://example.org/types/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .


task:task-1 a type:task ;
    relation:directlyPrecedes task:task-2 ;
    relation:instanceOf activity:W_Call%20after%20offers ;
    relation:performedBy resource:User_1 ;
    relation:partOf case:case-0 .
    
task:task-2 a type:task ;
    relation:directlyPrecedes task:task-3 ;
    relation:instanceOf activity:W_Validate%20application ;
    relation:performedBy resource:User_2 ;
    relation:partOf case:case-0 .
    
task:task-3 a type:task ;
    relation:instanceOf activity:W_Assess%20potential%20fraud ;
    relation:partOf case:case-0 .
    

activity:W_Handle%20leads a type:activity ;
    relation:canBeExecutedBy resource:User_2,
        resource:User_1,
        resource:User_3 .

activity:W_Complete%20application a type:activity ;
    relation:canBeExecutedBy resource:User_2,
        resource:User_1,
        resource:User_3 .


activity:W_Call%20after%20offers a type:activity ;
    relation:canBeExecutedBy resource:User_2,
        resource:User_1,
        resource:User_3 .

activity:W_Call%20incomplete%20files a type:activity ;
    relation:canBeExecutedBy resource:User_2,
        resource:User_1,
        resource:User_3 .

activity:W_Validate%20application a type:activity ;
    relation:canBeExecutedBy resource:User_2,
        resource:User_1,
        resource:User_3 .

activity:W_Assess%20potential%20fraud a type:activity ;
    relation:canBeExecutedBy resource:User_2,
        resource:User_1,
        resource:User_3 .

case:case-0 a type:case ;
    relation:ApplicationType ApplicationType:New%20credit ;
    relation:LoanGoal LoanGoal:Existing%20loan%20takeover .



ApplicationType:New%20credit a type:ApplicationType .

LoanGoal:Existing%20loan%20takeover a type:LoanGoal .

resource:User_3 a type:resource ;
    relation:available true.

resource:User_2 a type:resource ;
    relation:available true.

resource:User_1 a type:resource ;
    relation:available true.
    
four_eyes:rule1 a type:four_eyes ;
    relation:contains activity:W_Validate%20application, activity:W_Assess%20potential%20fraud.


'''

test_graph = ProcessKnowledgeGraph()
test_graph.parse(data=test_graph_data, format='turtle')
# draw_graph(test_graph)

In [ ]:
_uri = test_graph.uri
# test_graph.add(test_graph.entity_triple('RiskClass', 'Low'))
# test_graph.add(test_graph.entity_triple('RiskClass', 'Medium'))
# test_graph.add(test_graph.entity_triple('RiskClass', 'High'))
# test_graph.add(test_graph.entity_triple('Seniority', 'Low'))
# test_graph.add((_uri('RiskClass:Low'), test_graph.attribute_relation('minSeniority'), _uri('Seniority:Low')))
test_graph.add((_uri('resource:User_1'), test_graph.attribute_relation('Seniority'), _uri('Seniority:Low')))
# test_graph.add(test_graph.entity_triple('Seniority', 'Medium'))
# test_graph.add((_uri('RiskClass:Medium'), test_graph.attribute_relation('minSeniority'), _uri('Seniority:Medium')))
test_graph.add((_uri('resource:User_2'), test_graph.attribute_relation('Seniority'), _uri('Seniority:Medium')))
# test_graph.add(test_graph.entity_triple('Seniority', 'High'))
# test_graph.add((_uri('RiskClass:High'), test_graph.attribute_relation('minSeniority'), _uri('Seniority:High')))
test_graph.add((_uri('resource:User_3'), test_graph.attribute_relation('Seniority'), _uri('Seniority:High')))

# test_graph.parse('./src/extension_instances.ttl', format='n3')
# test_graph.add((_uri('Seniority:High'), test_graph.attribute_relation('greaterEq'), _uri('Seniority:High')))
# test_graph.add((_uri('Seniority:Medium'), test_graph.attribute_relation('greaterEq'), _uri('Seniority:Medium')))
# test_graph.add((_uri('Seniority:Low'), test_graph.attribute_relation('greaterEq'), _uri('Seniority:Low')))
# test_graph.add((_uri('Seniority:High'), test_graph.attribute_relation('greaterEq'), _uri('Seniority:Low')))
# test_graph.add((_uri('Seniority:High'), test_graph.attribute_relation('greaterEq'), _uri('Seniority:Medium')))
# test_graph.add((_uri('Seniority:Medium'), test_graph.attribute_relation('greaterEq'), _uri('Seniority:Low')))

# test_graph.add((_uri('LoanGoal:Existing loan takeover'), test_graph.attribute_relation('RiskClass'), _uri('RiskClass:Medium')))
draw_graph(test_graph)

In [ ]:

task_to_be_allocated = test_graph.uri('task:task-3')

validator = SHACLAllocator(test_graph)
validator.shacl_graph = Graph() # reset
# validator.load_extension(rules_ext='./src/extension_riskReqSeniority.ttl')

validator.use_hypothetical = True
validator.load_extension(rules_ext='./src/extension_riskReqSeniority.ttl')
validator.load_extension(instance_ext='./src/extension_instances.ttl')

for sen_index, seniority in enumerate(['Low', 'Medium', 'High']):
    test_graph.set((_uri('LoanGoal:Existing loan takeover'), test_graph.attribute_relation('RiskClass'), _uri('RiskClass:'+seniority)))
    for res_index, resource in enumerate(['User_1', 'User_2', 'User_3']):
        print(test_graph.uri('resource:'+resource))
        print(validator.test_assignment(task_to_be_allocated, test_graph.uri('resource:'+resource))[2])

In [ ]:
# print(test_graph.serialize(format='n3'))

In [ ]:
import random
x = '\n\n'.join(list(map(lambda goal : f'LoanGoal:{quote(goal)} a type:LoanGoal ;\n    relation:RiskClass RiskClass:{random.sample(['Low', 'Medium', 'High'], 1)[0]} .', simulator.problem.data_types['LoanGoal']._values)))
print(x)



In [ ]:
test_graph = ProcessKnowledgeGraph()
test_graph.parse(data=test_graph_data, format='turtle')
validator = SHACLAllocator(test_graph)
validator.shacl_graph = Graph() # reset

validator.load_extension(rules_ext='./src/extension_riskReqSeniority.ttl')
validator.load_extension(instance_ext='./src/extension_instances.ttl')

test_graph.add((test_graph.uri('resource:User_1'), test_graph.attribute_relation('Seniority'), test_graph.uri('Seniority:Low')))
test_graph.add((test_graph.uri('resource:User_2'), test_graph.attribute_relation('Seniority'), test_graph.uri('Seniority:Medium')))
test_graph.add((test_graph.uri('resource:User_3'), test_graph.attribute_relation('Seniority'), test_graph.uri('Seniority:High')))


task_to_be_allocated = test_graph.uri('task:task-3')
for sen_index, seniority in enumerate(['Low', 'Medium', 'High']):
    test_graph.set((test_graph.uri('LoanGoal:Existing loan takeover'), test_graph.attribute_relation('RiskClass'), test_graph.uri('RiskClass:'+seniority)))
    for res_index, resource in enumerate(['User_1', 'User_2', 'User_3']):
        test_result = validator.test_assignment(task_to_be_allocated, test_graph.uri('resource:'+resource))
        score, verdict = validator.calculate_score(test_result)
        if res_index >= sen_index:
            assert score > 0 
        else:
            assert score < 0 and verdict != '' 
        print((score, verdict))